---
title: "Spark SQL Job Data Analysis"
author: "Zhengyu Zhou"
format: html
embed-resources: true
date: "2025-10-18"
date-format: long
execute:
  echo: true
---

In [13]:
from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", "\"").csv("data/job_postings.csv")

# 3. Register the DataFrame as a temporary SQL table
df.createOrReplaceTempView("jobs")

In [14]:
# Verify the Data

# Display the first five rows
df.show(5)

# Show the schema (column names & data types)
df.printSchema()

+--------------------+-----------------+----------------------+----------+----------+----------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+--------------+-------------------+--------------+-------------------+---------------+--------------------+--------------------+--------------------+-------------+-------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-------+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+----

In [26]:
import pandas as pd

# read .xlsx and convert to .csv
excel_path = "data/job_postings.xlsx"   
csv_path   = "data/job_postings.csv"

# csv file
df = pd.read_excel(excel_path, engine="openpyxl")
df.to_csv(csv_path, index=False)

print("Converted", csv_path)
print("len:", len(df), "columns:", len(df.columns))

Converted data/job_postings.csv
len: 72476 columns: 123


In [27]:
#How many job postings we have in the dataset?
total_jobs = spark.sql("""
    SELECT COUNT(*) AS total_postings
    FROM jobs
""")
total_jobs.show()

+--------------+
|total_postings|
+--------------+
|         72476|
+--------------+



We have 72,476 total postings.

In [28]:
#Find the top 5 most common job titles
top_titles = spark.sql("""
    SELECT TITLE_NAME, COUNT(*) AS job_count
    FROM jobs
    WHERE TITLE_NAME IS NOT NULL
                       AND TITLE_NAME !='Unclassified'
    GROUP BY TITLE_NAME
    ORDER BY job_count DESC
    LIMIT 5
""")
top_titles.show(truncate=False)

+------------------------------+---------+
|TITLE_NAME                    |job_count|
+------------------------------+---------+
|Data Analysts                 |8593     |
|Business Intelligence Analysts|2074     |
|Enterprise Architects         |1999     |
|Oracle Cloud HCM Consultants  |1042     |
|Data Modelers                 |668      |
+------------------------------+---------+



Top 5 most common job titles
Data Analysts
Business Intelligence Analysts
Enterprise Architects 
Oracle Cloud HCM Consultants
Data Modelers